<a href="https://colab.research.google.com/github/Rafi068/Automated-Concrete-Mix-Design-/blob/main/Missing_Narayanganj_PM25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import io

print("Please upload your 'Narayanganj_PM25.xlsx' file:")
uploaded = files.upload()

# Get the actual uploaded filename, which might be renamed by Colab
actual_filename = list(uploaded.keys())[0]

if 'Narayanganj_PM25.xlsx' in actual_filename:
    print(f"File '{actual_filename}' uploaded successfully!")
else:
    print(f"Warning: You uploaded '{actual_filename}'. Please ensure you intended to upload a file named 'Narayanganj_PM25.xlsx'.")

Please upload your 'Narayanganj_PM25.xlsx' file:


Saving Narayanganj_PM25.xlsx to Narayanganj_PM25.xlsx
File 'Narayanganj_PM25.xlsx' uploaded successfully!


In [ ]:
import pandas as pd

df = pd.read_excel(io.BytesIO(uploaded[actual_filename]))
print('Original DataFrame Info:')
df.info()
display(df.head())

Original DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8759 entries, 0 to 8758
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          8759 non-null   datetime64[ns]
 1   PM2.5         8753 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 137.0 KB


,Date,PM2.5
0,2021-01-01 01:00:00,232.73
1,2021-01-01 02:00:00,238.40
2,2021-01-01 03:00:00,221.15
3,2021-01-01 04:00:00,214.25
4,2021-01-01 05:00:00,166.73


In [ ]:
import pandas as pd
import io

# Load all sheets from the Excel file
# The 'uploaded' variable contains the byte content of the file from the previous step
xls = pd.ExcelFile(io.BytesIO(uploaded[actual_filename]))

all_years_data = []

# Iterate through each sheet (year)
for year in xls.sheet_names:
    temp_df = xls.parse(year)
    # Rename columns to standard names
    temp_df.columns = ['datetime', 'PM2.5_Value']
    temp_df['Year'] = int(year) # Add a 'Year' column based on sheet name
    all_years_data.append(temp_df)

# Concatenate all yearly data into a single DataFrame
df_combined = pd.concat(all_years_data, ignore_index=True)

# Ensure 'datetime' column is in datetime format
# The user specified 'month/day/year hour' format, e.g., '1/1/2021 1:00'
df_combined['datetime'] = pd.to_datetime(df_combined['datetime'], format='%m/%d/%Y %H:%M', errors='coerce')

# Extract components for easier grouping and imputation
df_combined['Month'] = df_combined['datetime'].dt.month
df_combined['Day'] = df_combined['datetime'].dt.day
df_combined['Hour'] = df_combined['datetime'].dt.hour

# Sort the DataFrame by year, month, day, and hour for proper imputation
df_combined = df_combined.sort_values(by=['Year', 'Month', 'Day', 'Hour']).reset_index(drop=True)

print("Original DataFrame Info after combining all sheets:")
df_combined.info()
print("\nMissing values before imputation:")
print(df_combined.isnull().sum())
display(df_combined.head())
display(df_combined.tail())

Original DataFrame Info after combining all sheets:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26277 entries, 0 to 26276
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   datetime     26277 non-null  datetime64[ns]
 1   PM2.5_Value  26259 non-null  float64       
 2   Year         26277 non-null  int64         
 3   Month        26277 non-null  int32         
 4   Day          26277 non-null  int32         
 5   Hour         26277 non-null  int32         
dtypes: datetime64[ns](1), float64(1), int32(3), int64(1)
memory usage: 923.9 KB

Missing values before imputation:
datetime        0
PM2.5_Value    18
Year            0
Month           0
Day             0
Hour            0
dtype: int64


,datetime,PM2.5_Value,Year,Month,Day,Hour
0,2021-01-01 01:00:00,232.73,2021,1,1,1
1,2021-01-01 02:00:00,238.40,2021,1,1,2
2,2021-01-01 03:00:00,221.15,2021,1,1,3
3,2021-01-01 04:00:00,214.25,2021,1,1,4
4,2021-01-01 05:00:00,166.73,2021,1,1,5


,datetime,PM2.5_Value,Year,Month,Day,Hour
26272,2023-12-31 19:00:00,127.75,2023,12,31,19
26273,2023-12-31 20:00:00,142.24,2023,12,31,20
26274,2023-12-31 21:00:00,139.05,2023,12,31,21
26275,2023-12-31 22:00:00,127.41,2023,12,31,22
26276,2023-12-31 23:00:00,132.11,2023,12,31,23


In [ ]:
# --- Imputation Logic ---
df_imputed = df_combined.copy()

# Identify values that are initially NaN to create the imputed_flag later
original_na_mask = df_imputed['PM2.5_Value'].isna()

# Rule 2: If more than 10 hourly values are present in a day, fill missing values for that day.
def impute_day_trend(group):
    non_null_count = group['PM2.5_Value'].count()
    if non_null_count > 10:
        # Interpolate missing values within the day
        # 'limit_direction='both'' ensures interpolation works forward and backward
        # 'limit_area='inside'' ensures it doesn't extrapolate if the first/last values are NaN
        group['PM2.5_Value'] = group['PM2.5_Value'].interpolate(method='linear', limit_direction='both', limit_area='inside')
    return group

df_imputed = df_imputed.groupby(['Year', 'Month', 'Day'], group_keys=False).apply(impute_day_trend)

print('\nMissing values after Rule 2 (within-day) imputation:')
print(df_imputed.isnull().sum())

# Rule 3: If the data of the whole month is missing in a year, then follow the trend of other years.
# This also helps with days that had <= 10 non-null values (Rule 2 ignored them) or days with all NaNs that were missed.

# Calculate the mean PM2.5_Value for each Month-Day-Hour combination across all years
# This average will be used to fill remaining NaNs
mean_across_years = df_imputed.groupby(['Month', 'Day', 'Hour'])['PM2.5_Value'].mean()

# To apply the mean values to fill remaining NaNs, we need to align the mean_across_years Series
# with the df_imputed DataFrame based on their common index ['Month', 'Day', 'Hour'].
# Temporarily set the index of df_imputed to ['Month', 'Day', 'Hour']
df_imputed_indexed = df_imputed.set_index(['Month', 'Day', 'Hour'])

# Fill NaNs in 'PM2.5_Value' using the aligned mean_across_years Series
df_imputed_indexed['PM2.5_Value'] = df_imputed_indexed['PM2.5_Value'].fillna(mean_across_years)

# Reset the index to restore the original DataFrame structure
df_imputed = df_imputed_indexed.reset_index()

print('\nMissing values after Rule 3 (across-year) imputation:')
print(df_imputed.isnull().sum())

display(df_imputed.head())
display(df_imputed.tail())

# Add the 'imputed_flag' column
# Values that were originally NaN but are now not NaN are considered imputed
df_imputed['imputed_flag'] = original_na_mask & ~df_imputed['PM2.5_Value'].isna()

# Output the result
print("\nDataFrame with imputed values and 'imputed_flag':")
display(df_imputed.head(10))
display(df_imputed.tail(10))

# Save the imputed DataFrame to a CSV file
output_filename = 'Narayanganj_PM25_imputed.csv'
df_imputed.to_csv(output_filename, index=False)
print(f"\nImputed DataFrame saved to {output_filename}")

from google.colab import files
files.download(output_filename)


Missing values after Rule 2 (within-day) imputation:
datetime       0
PM2.5_Value    0
Year           0
Month          0
Day            0
Hour           0
dtype: int64

Missing values after Rule 3 (across-year) imputation:
Month          0
Day            0
Hour           0
datetime       0
PM2.5_Value    0
Year           0
dtype: int64


/tmp/ipykernel_5746/1131434901.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_imputed = df_imputed.groupby(['Year', 'Month', 'Day'], group_keys=False).apply(impute_day_trend)


,Month,Day,Hour,datetime,PM2.5_Value,Year
0,1,1,1,2021-01-01 01:00:00,232.73,2021
1,1,1,2,2021-01-01 02:00:00,238.40,2021
2,1,1,3,2021-01-01 03:00:00,221.15,2021
3,1,1,4,2021-01-01 04:00:00,214.25,2021
4,1,1,5,2021-01-01 05:00:00,166.73,2021


,Month,Day,Hour,datetime,PM2.5_Value,Year
26272,12,31,19,2023-12-31 19:00:00,127.75,2023
26273,12,31,20,2023-12-31 20:00:00,142.24,2023
26274,12,31,21,2023-12-31 21:00:00,139.05,2023
26275,12,31,22,2023-12-31 22:00:00,127.41,2023
26276,12,31,23,2023-12-31 23:00:00,132.11,2023



DataFrame with imputed values and 'imputed_flag':


,Month,Day,Hour,datetime,PM2.5_Value,Year,imputed_flag
0,1,1,1,2021-01-01 01:00:00,232.73,2021,False
1,1,1,2,2021-01-01 02:00:00,238.40,2021,False
2,1,1,3,2021-01-01 03:00:00,221.15,2021,False
3,1,1,4,2021-01-01 04:00:00,214.25,2021,False
4,1,1,5,2021-01-01 05:00:00,166.73,2021,False
5,1,1,6,2021-01-01 06:00:00,171.60,2021,False
6,1,1,7,2021-01-01 07:00:00,177.75,2021,False
7,1,1,8,2021-01-01 08:00:00,158.63,2021,False
8,1,1,9,2021-01-01 09:00:00,167.71,2021,False
9,1,1,10,2021-01-01 10:00:00,172.15,2021,False


,Month,Day,Hour,datetime,PM2.5_Value,Year,imputed_flag
26267,12,31,14,2023-12-31 14:00:00,126.94,2023,False
26268,12,31,15,2023-12-31 15:00:00,133.22,2023,False
26269,12,31,16,2023-12-31 16:00:00,130.93,2023,False
26270,12,31,17,2023-12-31 17:00:00,132.12,2023,False
26271,12,31,18,2023-12-31 18:00:00,129.55,2023,False
26272,12,31,19,2023-12-31 19:00:00,127.75,2023,False
26273,12,31,20,2023-12-31 20:00:00,142.24,2023,False
26274,12,31,21,2023-12-31 21:00:00,139.05,2023,False
26275,12,31,22,2023-12-31 22:00:00,127.41,2023,False
26276,12,31,23,2023-12-31 23:00:00,132.11,2023,False



Imputed DataFrame saved to Narayanganj_PM25_imputed.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

!pip install xlsxwriter

# Create an Excel writer object
output_excel_filename = 'Narayanganj_PM25_imputed_yearly.xlsx'
writer = pd.ExcelWriter(output_excel_filename, engine='xlsxwriter')

# Get unique years from the imputed DataFrame
years = df_imputed['Year'].unique()

for year in sorted(years):
    # Filter data for the current year
    df_year = df_imputed[df_imputed['Year'] == year]

    # Write the yearly DataFrame to a separate sheet
    # The sheet name will be the year (e.g., '2021', '2022', '2023')
    df_year.to_excel(writer, sheet_name=str(year), index=False)

# Save the Excel file
writer.close()

print(f"Excel file with yearly sheets saved to {output_excel_filename}")

# Optionally, download the file
from google.colab import files
files.download(output_excel_filename)

  Using cached xlsxwriter-3.2.9-py3-none-any.whl.metadata (2.7 kB)
Using cached xlsxwriter-3.2.9-py3-none-any.whl (175 kB)
Excel file with yearly sheets saved to Narayanganj_PM25_imputed_yearly.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>